# OmniRoute gateway (Phase GW) — fail-closed startup gate

> Notebook generated from [`examples/gateways_omniroute.py`](https://github.com/prismal-ai/prismal/blob/main/examples/gateways_omniroute.py).

| Field | Value |
|------|-------|
| **API key** | 🔑 Not required — the health probe is an `httpx.MockTransport` |
| **Source** | `examples/gateways_omniroute.py` |
| **Guide** | [`docs/gateways.md`](https://github.com/prismal-ai/prismal/blob/main/docs/gateways.md) |

OmniRoute is **self-hosted**, so prismal will not trust it blindly. Before any model can be resolved the router
runs a startup gate that needs an API key, an upstream attestation (`omniroute_upstream_attestation`) and a
version floor for CVE-2026-88062. Until `astartup()` succeeds — or if the gate itself crashes — OmniRoute stays
disabled (fail closed).

1. Version 3.8.50 (vulnerable) is refused and the router disables the gateway.
2. Version 3.8.51 (fixed) passes and the gateway resolves models.

In [1]:
# Enable nested event loop to use `await` directly in Jupyter.
import sys
if 'nest_asyncio' not in sys.modules:
    try:
        import nest_asyncio
        nest_asyncio.apply()
    except ImportError:
        %pip install -q nest_asyncio
        import nest_asyncio
        nest_asyncio.apply()

## Setup · a gateway factory with a fake `/api/monitoring/health`

In [2]:
from __future__ import annotations

import httpx
from pydantic import SecretStr

from prismal.core.config import Settings
from prismal.core.exceptions import GatewayPolicyError
from prismal.providers.gateways.omniroute import OmniRouteGateway
from prismal.providers.gateways.router import GatewayRouter


def make_gateway(settings: Settings, version: str) -> OmniRouteGateway:
    def health(request: httpx.Request) -> httpx.Response:
        return httpx.Response(200, json={"version": version})

    return OmniRouteGateway(
        settings=settings,
        secret_resolver=lambda ref: SecretStr("omni-demo-key"),
        http=httpx.AsyncClient(transport=httpx.MockTransport(health)),
    )


settings = Settings(
    llm_gateways_enabled=True,
    omniroute_enabled=True,
    omniroute_upstream_attestation="api_keys_only",
)

## 1 · OmniRoute 3.8.50 (affected by CVE-2026-88062)

In [3]:
router = GatewayRouter({"omniroute": make_gateway(settings, "3.8.50")}, settings=settings)
await router.astartup()
print("disabled:", router.disabled)
try:
    router.resolve("gw:omniroute/my-combo", agent=None, session_id=None)
except GatewayPolicyError as exc:
    print("resolve refused:", exc.reason)
await router.aclose()

[info     ] gateway.policy_denied          agent=None gateway=omniroute reason=omniroute_vulnerable_version


[warning  ] gateway.disabled_at_startup    gateway=omniroute reason=omniroute_vulnerable_version


disabled: {'omniroute': 'omniroute_vulnerable_version'}
resolve refused: omniroute_vulnerable_version


## 2 · OmniRoute 3.8.51 (fixed)

In [4]:
router = GatewayRouter({"omniroute": make_gateway(settings, "3.8.51")}, settings=settings)
await router.astartup()
resolved = router.resolve("gw:omniroute/my-combo", agent=None, session_id=None)
assert resolved is not None
print("litellm model:", resolved.litellm_model, "| api_base:", resolved.api_base)
await router.aclose()

litellm model: openai/my-combo | api_base: http://localhost:20128/v1


---

In a host, `build_runtime(..., gateway_vault=...)` composes `RuntimeContext.gateways` per tenant and runs `astartup()` for you.